In [2]:
from utils import readJson, writeJson

In [3]:
dataset = readJson('Dataset/transfermarkt_fbref_dataset.json')
dataset_manc = readJson('Dataset/transfermarkt_fbref_mancanti.json')
#merge giocatori mancanti con dataset originale
for i in range(len(dataset_manc)):
    for j in range(len(dataset)):
        if dataset_manc[i]['stats'] != {} and dataset_manc[i]['id'] == dataset[j]['id']:
            dataset[j] = dataset_manc[i]

leagues = ['seriea', 'premierleague', 'ligue1', 'laliga', 'bundesliga']
transfers = []
for l in leagues:
    transfers+= readJson(f'Dataset/Transfermarkt/Transfers/{l}_2024.json')
#levo rientri dal prestito
transfers = [x for x in transfers if 'Fine prestito' not in x['cost']]
teams = set(x['tm_team'].upper() for x in dataset)

In [4]:
def computeTransferEvent(transf, dataset, teams, pos_en):
    event = {}
    for play in dataset:
        if transf['player_id'] == play['tm_id']:
            if transf['team_buyer'].upper() not in teams:
                return {}
            if play['tm_team'].upper() == transf['team_buyer'].upper():
                return event
            event['id'] = play['id']
            event['player_name'] = play['name']
            try:
                if play['tm_role'] == 'Portiere':
                    return {}
                
                event['position'] = play['position']
            except:
                #print(play['name'])
                return {}
            
            event['team'] = transf['team_buyer']
            event['tm_role'] = play['tm_role']
            event['tm_role_en'] = pos_en[play['tm_role']]
            event['cost'] = transf['cost']

    return event

In [5]:
transf_events = []
position_tr = readJson('Dataset/tm_position_translation.json')
i=0
log_transfer = {}
for transf in transfers:
    event = computeTransferEvent(transf, dataset,teams, position_tr)
    if event != {}:
        if event['id'] not in log_transfer.keys():
            transf_events.append(event)
            log_transfer[event['id']] = event['team']
        '''
        else:
            if event['team'] == log_transfer[event['id']]:
                #print(f"{event['player_name']}: record duplicato")
                continue
            else:
                print(event, log_transfer[event[id]])
        '''
writeJson(transf_events, 'Dataset/log_trasferimenti_gt.json')
len(transf_events)

300

In [7]:
ruoli = {}
for t in transf_events:
    r = t['tm_role']
    if r in ruoli.keys():
        ruoli[r] = ruoli[r] +1
    else:
        ruoli[r] = 1

ruoli

{'Centrocampista di sinistra': 3,
 'Centrale': 41,
 'Punta centrale': 50,
 'Trequartista': 14,
 'Mediano': 28,
 'Terzino destro': 26,
 'Seconda punta': 5,
 'Ala sinistra': 24,
 'Ala destra': 23,
 'Terzino sinistro': 15,
 'Difensore centrale': 67,
 'Centrocampista di destra': 4}

In [8]:
ruoli = {}
for t in dataset:
    r = t['tm_role']
    if r in ruoli.keys():
        ruoli[r] = ruoli[r] +1
    else:
        ruoli[r] = 1

ruoli

{'Difensore centrale': 428,
 'Ala sinistra': 181,
 'Centrale': 319,
 'Ala destra': 167,
 'Portiere': 136,
 'Terzino destro': 200,
 'Punta centrale': 320,
 'Mediano': 180,
 'Terzino sinistro': 175,
 'Trequartista': 150,
 'Seconda punta': 14,
 'Centrocampista di sinistra': 19,
 'Centrocampista di destra': 17}

In [26]:
recs = readJson('Descriptions/Descriptions/recommendations.json')

def get_ids_recommendation(rec):
    ids = []
    for id_str in rec['recommendation'].split('- ID: '):
        id = id_str[:8]
        ids.append(id)
    return ids[1:]

def get_ids_retrieval(rec):
    ids = []
    for id_str in rec['prompt'].split('- ID: '):
        id = id_str[:8]
        ids.append(id)
    return ids[2:]


hit_rec = 0
hit_ret = 0
for rec in recs:
    id = rec['id'] 
    ids_rec = get_ids_recommendation(rec)
    if id in ids_rec:
        hit_rec+=1
    ids_ret = get_ids_retrieval(rec)
    if id in ids_ret:
        print(rec['player_name'])
        hit_ret+=1
    #print('recommendation: ', id, ids_rec)
    #print('retrieval: ', id, ids_ret)
hit_rec, hit_ret

        

Emerson


(0, 1)

In [37]:
eval = readJson('Descriptions/Descriptions/evaluation_90.json')
ret_count = 0
rec_count = 0
for k, sym in eval['similarity'].items():
    recs = list(sym['recommendation'].values())
    if recs.count(1) > recs.count(0):
        rec_count+=1
    rets = list(sym['retrieval'].values())
    if rets.count(1) > rets.count(0):
        ret_count+=1

ret_count, rec_count

(133, 120)

In [39]:
eval = readJson('Descriptions/Descriptions/evaluation.json')
ret_count = 0
rec_count = 0
for k, sym in eval['similarity'].items():
    recs = list(sym['recommendation'].values())
    if recs.count(1) > recs.count(0):
        rec_count+=1
    rets = list(sym['retrieval'].values())
    if rets.count(1) > rets.count(0):
        ret_count+=1

ret_count, rec_count

(263, 252)

In [42]:
eval = readJson('Descriptions/Descriptions/evaluation_cf.json')
ret_count = 0
rec_count = 0
for k, sym in eval['similarity_90'].items():
    recs = list(sym['recommendation'].values())
    if recs.count(1) > recs.count(0):
        rec_count+=1
    rets = list(sym['retrieval'].values())
    if rets.count(1) > rets.count(0):
        ret_count+=1

print('Similarity_90:',ret_count, rec_count)

ret_count = 0
rec_count = 0
for k, sym in eval['similarity_80'].items():
    recs = list(sym['recommendation'].values())
    if recs.count(1) > recs.count(0):
        rec_count+=1
    rets = list(sym['retrieval'].values())
    if rets.count(1) > rets.count(0):
        ret_count+=1

print('Similarity_80:',ret_count, rec_count)

Similarity_90: 131 120
Similarity_80: 265 251


In [ ]:
_